In [3]:
pip install transformers datasets evaluate accelerate scikit-learn torch

  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [5]:
# ==========================================================
# ADVANCED TASK 1 - NEWS TOPIC CLASSIFIER USING BERT
# OPTIMIZED FOR T4 GPU
# ==========================================================

# ==========================================================
# INSTALL REQUIRED LIBRARIES
# ==========================================================

# Uncomment if running first time

# !pip install transformers datasets evaluate accelerate -q

# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import numpy as np
import torch
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ==========================================================
# CHECK GPU
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

# ==========================================================
# LOAD DATASET
# ==========================================================

dataset = load_dataset("ag_news")

print(dataset)

# ==========================================================
# LABEL NAMES
# ==========================================================

label_names = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

# ==========================================================
# LOAD TOKENIZER
# ==========================================================

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

# ==========================================================
# TOKENIZATION FUNCTION
# ==========================================================

def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# ==========================================================
# TOKENIZE DATASET
# ==========================================================

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

# ==========================================================
# LOAD MODEL
# ==========================================================

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

model.to(device)

# ==========================================================
# EVALUATION METRICS
# ==========================================================

accuracy_metric = evaluate.load("accuracy")

f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

# ==========================================================
# SMALLER DATASET FOR FAST TRAINING
# ==========================================================

train_dataset = tokenized_dataset["train"] \
    .shuffle(seed=42) \
    .select(range(3000))

eval_dataset = tokenized_dataset["test"] \
    .select(range(500))

# ==========================================================
# TRAINING ARGUMENTS
# ==========================================================

training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch", # Changed from evaluation_strategy

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=50,

    fp16=True,                 # Faster on T4 GPU

    report_to="none"
)

# ==========================================================
# TRAINER
# ==========================================================

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=eval_dataset,

    compute_metrics=compute_metrics
)

# ==========================================================
# TRAIN MODEL
# ==========================================================

print("\nStarting Training...\n")

trainer.train()

# ==========================================================
# EVALUATE MODEL
# ==========================================================

results = trainer.evaluate()

print("\nEvaluation Results:")
print(results)

# ==========================================================
# SAVE MODEL
# ==========================================================

model.save_pretrained("./bert_news_classifier")

tokenizer.save_pretrained("./bert_news_classifier")

print("\nModel Saved Successfully!")

# ==========================================================
# TEST CUSTOM NEWS
# ==========================================================

sample_news = """
Apple launches new AI-powered iPhone with advanced chip technology.
"""

inputs = tokenizer(
    sample_news,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model(**inputs)

prediction = outputs.logits.argmax(dim=-1).item()

print("\nPredicted Category:")
print(label_names[prediction])

Using Device: cuda
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 


Starting Training...



Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.388433,0.396722,0.880000,0.880066


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluation Results:
{'eval_loss': 0.39672181010246277, 'eval_accuracy': 0.88, 'eval_f1': 0.8800657951607002, 'eval_runtime': 1.0727, 'eval_samples_per_second': 466.112, 'eval_steps_per_second': 29.831, 'epoch': 1.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model Saved Successfully!

Predicted Category:
Sci/Tech
